# VisionBridge — trained model checker (Colab)

This notebook is **inference-only**. It does not train, extract a dataset, or require MediaPipe.

Goal: verify that an already-trained `base_model.pt` + vocabulary can load, pass a forward test, and produce real decoded text from real 132-dim pose + 1404-dim face keypoint sequences.

A raw-video test is optional and isolated at the end. The core model check never installs MediaPipe.


## 1. Runtime check — no heavy reinstall

We intentionally do **not** uninstall or pin MediaPipe/TensorFlow/PyTorch here. The trained-model check only needs PyTorch and the VisionBridge Python modules.


In [ ]:
import os, sys, torch, platform
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


## 2. Load a fresh VisionBridge checkout

The notebook uses the current `main` branch so stale local code does not silently get used.


In [ ]:
import os, subprocess

REPO_ROOT = "/content/VisionBridge"
if os.path.isdir(REPO_ROOT):
    subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/BharathWaj-K-R/VisionBridge.git", REPO_ROOT], check=True)
os.chdir(REPO_ROOT)

print("REPO_ROOT:", REPO_ROOT)

with open("backend/app/models/base_model.py", encoding="utf-8") as f:
    model_src = f.read()
with open("backend/app/training/isltranslate.py", encoding="utf-8") as f:
    dataset_src = f.read()

assert "POSE_INPUT_DIM = 33 * 4" in model_src
assert "FACE_INPUT_DIM = 468 * 3" in model_src
assert "MAX_SEQUENCE_LENGTH = 1024" in model_src
assert "SimpleCharTokenizer" in dataset_src
print("VisionBridge model/data contracts verified: pose=132, face=1404, max_frames=1024.")


## 3. Locate the trained checkpoint and vocabulary

Preferred paths:

```text
backend/app/models/weights/base_model.pt
backend/app/models/weights/base_model.vocab.json
```

If they are not present, upload the two files from your Colab machine.


In [ ]:
from pathlib import Path

WEIGHTS = Path("backend/app/models/weights/base_model.pt")
VOCAB = Path("backend/app/models/weights/base_model.vocab.json")

if not WEIGHTS.exists() or not VOCAB.exists():
    print("Checkpoint/vocab not found in the repository checkout.")
    try:
        from google.colab import files
        uploaded = files.upload()
        for name in uploaded:
            src = Path(name)
            if name.endswith(".pt"):
                WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
                src.replace(WEIGHTS)
            elif name.endswith(".vocab.json") or name == "model.vocab.json":
                VOCAB.parent.mkdir(parents=True, exist_ok=True)
                src.replace(VOCAB)
    except ImportError:
        raise RuntimeError(
            "Missing base_model.pt/base_model.vocab.json. Put them at "
            f"{WEIGHTS} and {VOCAB} before continuing."
        )

assert WEIGHTS.exists(), f"Missing trained weights: {WEIGHTS}"
assert VOCAB.exists(), f"Missing vocabulary: {VOCAB}"
print("Weights:", WEIGHTS, f"({WEIGHTS.stat().st_size / 1024**2:.2f} MB)")
print("Vocab:  ", VOCAB, f"({VOCAB.stat().st_size / 1024:.2f} KB)")


## 4. Check checkpoint integrity before inference

This verifies the checkpoint is a real state dict and that its output head vocabulary matches the saved tokenizer.


In [ ]:
import torch
from app.training.isltranslate import SimpleCharTokenizer

tokenizer = SimpleCharTokenizer.load(VOCAB)
state = torch.load(WEIGHTS, map_location="cpu")

assert isinstance(state, dict), "Checkpoint is not a state-dict dictionary."
assert "output_head.weight" in state, "Checkpoint is missing output_head.weight."

checkpoint_vocab_size = int(state["output_head.weight"].shape[0])
print("Checkpoint vocabulary size:", checkpoint_vocab_size)
print("Tokenizer vocabulary size: ", tokenizer.vocab_size)
assert checkpoint_vocab_size == tokenizer.vocab_size, (
    f"Vocabulary mismatch: checkpoint={checkpoint_vocab_size}, "
    f"tokenizer={tokenizer.vocab_size}"
)

print("Checkpoint integrity: PASS")


## 5. Load the real trained model and run a forward smoke test

This checks that the actual trained checkpoint loads and that the output dimensions are compatible with CTC decoding.

The synthetic tensor here is **only a shape/integrity test**. It is not an accuracy test.


In [ ]:
from app.models.base_model import load_frozen_base_model, POSE_INPUT_DIM, FACE_INPUT_DIM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_frozen_base_model(str(WEIGHTS), vocab_size=tokenizer.vocab_size).to(device)

assert not any(p.requires_grad for p in model.parameters()), "Base model is not frozen."
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

frames = 16
pose = torch.zeros(1, frames, POSE_INPUT_DIM, device=device)
face = torch.zeros(1, frames, FACE_INPUT_DIM, device=device)

with torch.inference_mode():
    logits = model(pose, face)

print("Synthetic input pose:", tuple(pose.shape))
print("Synthetic input face:", tuple(face.shape))
print("Model logits:", tuple(logits.shape))

assert logits.shape[0] == 1
assert logits.shape[1] == frames
assert logits.shape[2] == tokenizer.vocab_size
print("Forward smoke test: PASS")


## 6. Real-data prediction test — use an actual keypoint sample

This is the primary accuracy sanity check.

The notebook first looks for `data/processed/isltranslate/` with `ISLTranslate.csv`, `pose/<uid>.npy`, and `face/<uid>.npy`. If present, it runs the trained model on real samples and prints ground truth, prediction, confidence, and CER.

If the processed dataset is unavailable, the notebook does not fabricate inputs; use the optional raw-video check below.


In [ ]:
from pathlib import Path

from app.services.inference_service import decode_logits
from app.training.isltranslate import ISLTranslateKeypointDataset, _downsample_to_max_length

DATA_DIR = Path("data/processed/isltranslate")

def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (0 if ca == cb else 1))
        prev = cur
    return prev[-1]

def cer(pred: str, truth: str) -> float:
    return levenshtein(pred.lower(), truth.lower()) / max(len(truth), 1)

if (DATA_DIR / "ISLTranslate.csv").exists() and (DATA_DIR / "pose").exists() and (DATA_DIR / "face").exists():
    dataset = ISLTranslateKeypointDataset(DATA_DIR, tokenizer=tokenizer)
    assert len(dataset) > 0, "Processed dataset exists but contains no usable samples."
    sample_indices = sorted(set([0, min(1, len(dataset)-1), len(dataset)//2, len(dataset)-1]))
    print(f"Testing {len(sample_indices)} real dataset samples out of {len(dataset)}.")
    with torch.inference_mode():
        for idx in sample_indices:
            item = dataset[idx]
            pose, face = _downsample_to_max_length(item["pose"], item["face"], item["uid"])
            logits = model(pose.unsqueeze(0).to(device), face.unsqueeze(0).to(device))
            predicted, confidence = decode_logits(logits)
            truth = item["text"]
            print("\n---", item["uid"])
            print("GROUND TRUTH :", truth)
            print("PREDICTED    :", predicted)
            print("CONFIDENCE   :", round(float(confidence), 4))
            print("CER          :", round(cer(predicted, truth), 4))
else:
    print("No processed keypoint dataset found; skipping labeled-data inference.")
    print("Use the optional raw-video check for a direct real-input test.")

print("\nMODEL CHECK COMPLETE")


## 7. Optional raw-video check (MediaPipe only needed here)

This section tests `video → MediaPipe → 132/1404 keypoints → trained model → text`. It is deliberately isolated so a MediaPipe installation problem cannot block the trained-checkpoint test above.

On Python 3.13, the old `mediapipe==0.10.21` pin is not usable; the current optional path selects a newer 0.10.x release for Python 3.13.


In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("mediapipe") is None:
    package = "mediapipe==0.10.35" if sys.version_info >= (3, 13) else "mediapipe==0.10.21"
    print("Installing optional video-test dependency:", package)
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", package])
    if result.returncode != 0:
        raise RuntimeError("Optional MediaPipe installation failed; model/keypoint checks above remain valid.")

import mediapipe as mp
print("MediaPipe:", mp.__version__)
assert hasattr(mp, "solutions"), "Optional video test requires legacy mp.solutions.holistic."
print("Optional MediaPipe path ready.")


## 8. Final interpretation

PASS means the checkpoint loads, vocabulary matches, the base model is frozen, the 132/1404 input contract is correct, the decoder executes, and real input produces a prediction.

A smoke-test PASS does **not** prove model accuracy. Use labeled held-out data and CER/WER for that claim.
